# Experiment 5: Decision Tree and Random Forest — A Comparative Classification Study

**Course:** ICS1512 — Machine Learning Algorithms Laboratory
**Institution:** Sri Sivasubramaniya Nadar College of Engineering, Chennai
**Degree & Branch:** M.Tech (Integrated) Computer Science & Engineering, Semester V

**Objective**
- Implement a Decision Tree classifier.
- Extend the Decision Tree into a Random Forest ensemble model.
- Study the impact of hyperparameters on overfitting and generalization.
- Select optimal hyperparameters using 5-Fold Cross-Validation.
- Compare single-tree and ensemble-tree models.

**Dataset:** Wisconsin Diagnostic Breast Cancer Dataset — 569 samples, 30 numerical features,
target classes Malignant (M) / Benign (B). Loaded via `sklearn.datasets.load_breast_cancer`, which is
the exact UCI dataset bundled with scikit-learn (identical to the UCI archive version).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_curve, auc,
                              RocCurveDisplay)

sns.set_style("whitegrid")
%matplotlib inline
RANDOM_STATE = 42

## Step 1 — Load the Dataset and Encode Class Labels

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# target: 0 = malignant, 1 = benign (as encoded by sklearn); relabel for clarity
df["diagnosis"] = df["target"].map({0: "Malignant", 1: "Benign"})

print("Shape:", df.shape)
print("\nClass encoding -> 0: Malignant (M), 1: Benign (B)")
df.head()

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("\nFeature list (first 10):", list(data.feature_names[:10]), "...")
df.describe().T.head(10)

## Step 2 — Exploratory Data Analysis

In [ ]:
# Class distribution
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="diagnosis", palette=["#e74c3c", "#2ecc71"])
plt.title("Class Distribution: Malignant vs Benign")
plt.xlabel("Diagnosis"); plt.ylabel("Count")
plt.show()

print(df["diagnosis"].value_counts())
print(df["diagnosis"].value_counts(normalize=True).round(3) * 100, "%")

In [ ]:
# Feature correlation heatmap (first 15 features, to keep it readable)
plt.figure(figsize=(12, 10))
corr = df[data.feature_names[:15]].corr()
sns.heatmap(corr, annot=False, cmap="coolwarm", square=True)
plt.title("Feature Correlation Heatmap (first 15 features)")
plt.show()

In [ ]:
# Distribution of a few key discriminative features by class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ["mean radius", "mean texture", "mean concavity"]):
    sns.boxplot(data=df, x="diagnosis", y=feat, ax=ax, palette=["#e74c3c", "#2ecc71"])
    ax.set_title(f"{feat} by Diagnosis")
plt.tight_layout()
plt.show()

## Step 3 — Train/Test Split (80–20)

In [ ]:
X = df[data.feature_names]
y = df["target"]   # 0 = Malignant, 1 = Benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("\nTrain class balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:\n", y_test.value_counts(normalize=True).round(3))

## Step 4 — Train a Baseline Decision Tree Classifier

In [ ]:
dt_baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_baseline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, dt_baseline.predict(X_train))
test_acc = accuracy_score(y_test, dt_baseline.predict(X_test))
print(f"Baseline Decision Tree — Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")
print(f"Tree depth: {dt_baseline.get_depth()} | Leaves: {dt_baseline.get_n_leaves()}")
print("\n(Note the train/test gap — a sign of overfitting with an unconstrained deep tree.)")

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(dt_baseline, max_depth=2, feature_names=data.feature_names,
          class_names=["Malignant", "Benign"], filled=True, fontsize=9)
plt.title("Baseline Decision Tree (top 2 levels shown)")
plt.show()

## Step 5 & 6 — Decision Tree Hyperparameter Search Space + 5-Fold Cross-Validation

Hyperparameters explored: `criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf`.

In [ ]:
dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=dt_param_grid,
    cv=cv,
    scoring={"accuracy": "accuracy", "f1": "f1"},
    refit="accuracy",
    n_jobs=-1,
)
dt_grid.fit(X_train, y_train)

print("Total hyperparameter combinations evaluated:", len(dt_grid.cv_results_["params"]))
print("\nBest Decision Tree params:", dt_grid.best_params_)
print(f"Best avg CV accuracy: {dt_grid.best_score_:.4f}")

In [ ]:
dt_results = pd.DataFrame(dt_grid.cv_results_)
dt_results["Avg CV Accuracy (%)"] = (dt_results["mean_test_accuracy"] * 100).round(2)
dt_results["Avg CV F1 Score"] = dt_results["mean_test_f1"].round(4)
dt_results["criterion"] = dt_results["param_criterion"]
dt_results["max_depth"] = dt_results["param_max_depth"]

# Summarised table: best combination per (criterion, max_depth) pair
dt_summary = (dt_results
              .sort_values("mean_test_accuracy", ascending=False)
              .groupby(["criterion", "max_depth"], dropna=False)
              .first()
              .reset_index()[["criterion", "max_depth", "Avg CV Accuracy (%)", "Avg CV F1 Score"]]
              .sort_values("Avg CV Accuracy (%)", ascending=False)
              .reset_index(drop=True))

dt_summary.to_csv("dt_cv_summary.csv", index=False)
dt_summary.head(10)

## Step 7 — Select Best Decision Tree Hyperparameters and Retrain

In [ ]:
dt_best = dt_grid.best_estimator_
dt_best.fit(X_train, y_train)

dt_train_acc = accuracy_score(y_train, dt_best.predict(X_train))
dt_test_acc = accuracy_score(y_test, dt_best.predict(X_test))
print("Tuned Decision Tree params:", dt_grid.best_params_)
print(f"Train accuracy: {dt_train_acc:.4f} | Test accuracy: {dt_test_acc:.4f}")
print(f"Tree depth: {dt_best.get_depth()} | Leaves: {dt_best.get_n_leaves()}")

## Step 8 — Train a Random Forest Classifier

In [ ]:
rf_baseline = RandomForestClassifier(random_state=RANDOM_STATE)
rf_baseline.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf_baseline.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_baseline.predict(X_test))
print(f"Baseline Random Forest — Train accuracy: {rf_train_acc:.4f} | Test accuracy: {rf_test_acc:.4f}")

## Step 9 — Random Forest Hyperparameter Search Space + 5-Fold Cross-Validation

Hyperparameters explored: `n_estimators`, `max_depth`, `max_features`, `bootstrap`.

In [ ]:
rf_param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=rf_param_grid,
    cv=cv,
    scoring={"accuracy": "accuracy", "f1": "f1"},
    refit="accuracy",
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

print("Total hyperparameter combinations evaluated:", len(rf_grid.cv_results_["params"]))
print("\nBest Random Forest params:", rf_grid.best_params_)
print(f"Best avg CV accuracy: {rf_grid.best_score_:.4f}")

In [ ]:
rf_results = pd.DataFrame(rf_grid.cv_results_)
rf_results["Avg CV Accuracy (%)"] = (rf_results["mean_test_accuracy"] * 100).round(2)
rf_results["Avg CV F1 Score"] = rf_results["mean_test_f1"].round(4)
rf_results["n_estimators"] = rf_results["param_n_estimators"]
rf_results["max_depth"] = rf_results["param_max_depth"]
rf_results["max_features"] = rf_results["param_max_features"]

rf_summary = (rf_results
              .sort_values("mean_test_accuracy", ascending=False)
              .groupby(["n_estimators", "max_depth", "max_features"], dropna=False)
              .first()
              .reset_index()[["n_estimators", "max_depth", "max_features",
                               "Avg CV Accuracy (%)", "Avg CV F1 Score"]]
              .sort_values("Avg CV Accuracy (%)", ascending=False)
              .reset_index(drop=True))

rf_summary.to_csv("rf_cv_summary.csv", index=False)
rf_summary.head(10)

## Step 10 — Select Best Random Forest Hyperparameters and Retrain

In [ ]:
rf_best = rf_grid.best_estimator_
rf_best.fit(X_train, y_train)

rf_best_train_acc = accuracy_score(y_train, rf_best.predict(X_train))
rf_best_test_acc = accuracy_score(y_test, rf_best.predict(X_test))
print("Tuned Random Forest params:", rf_grid.best_params_)
print(f"Train accuracy: {rf_best_train_acc:.4f} | Test accuracy: {rf_best_test_acc:.4f}")

## Step 11 — Compare Both Models Using Evaluation Metrics

In [ ]:
def evaluate(model, X_te, y_te, name):
    pred = model.predict(X_te)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred),
        "Recall": recall_score(y_te, pred),
        "F1-score": f1_score(y_te, pred),
    }

metrics_df = pd.DataFrame([
    evaluate(dt_best, X_test, y_test, "Decision Tree (tuned)"),
    evaluate(rf_best, X_test, y_test, "Random Forest (tuned)"),
]).round(4)
metrics_df.to_csv("evaluation_metrics.csv", index=False)
metrics_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, model, name in zip(axes, [dt_best, rf_best], ["Decision Tree", "Random Forest"]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Malignant", "Benign"], yticklabels=["Malignant", "Benign"])
    ax.set_title(f"{name} — Confusion Matrix")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve and AUC
plt.figure(figsize=(6, 5))
for model, name, color in [(dt_best, "Decision Tree", "darkorange"), (rf_best, "Random Forest", "green")]:
    y_score = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Decision Tree vs Random Forest")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Feature importance comparison (top 10)
dt_imp = pd.Series(dt_best.feature_importances_, index=data.feature_names).sort_values(ascending=False).head(10)
rf_imp = pd.Series(rf_best.feature_importances_, index=data.feature_names).sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
dt_imp.sort_values().plot.barh(ax=axes[0], color="darkorange")
axes[0].set_title("Decision Tree — Top 10 Feature Importances")
rf_imp.sort_values().plot.barh(ax=axes[1], color="green")
axes[1].set_title("Random Forest — Top 10 Feature Importances")
plt.tight_layout()
plt.show()

## Step 12 — 5-Fold Cross-Validation Performance Comparison (Tuned Models)

In [ ]:
dt_fold_scores = cross_val_score(dt_best, X_train, y_train, cv=cv, scoring="accuracy")
rf_fold_scores = cross_val_score(rf_best, X_train, y_train, cv=cv, scoring="accuracy")

fold_comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Random Forest"],
    "Fold 1": [dt_fold_scores[0], rf_fold_scores[0]],
    "Fold 2": [dt_fold_scores[1], rf_fold_scores[1]],
    "Fold 3": [dt_fold_scores[2], rf_fold_scores[2]],
    "Fold 4": [dt_fold_scores[3], rf_fold_scores[3]],
    "Fold 5": [dt_fold_scores[4], rf_fold_scores[4]],
})
fold_comparison["Average"] = fold_comparison[["Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"]].mean(axis=1)
fold_comparison.iloc[:, 1:] = fold_comparison.iloc[:, 1:].round(4)
fold_comparison.to_csv("fold_comparison.csv", index=False)
fold_comparison

In [ ]:
plt.figure(figsize=(7, 4.5))
folds = ["Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"]
plt.plot(folds, dt_fold_scores, marker="o", label="Decision Tree", color="darkorange")
plt.plot(folds, rf_fold_scores, marker="o", label="Random Forest", color="green")
plt.ylabel("Accuracy"); plt.title("5-Fold CV Accuracy — Decision Tree vs Random Forest")
plt.legend(); plt.ylim(0.85, 1.0)
plt.show()

## Observation Questions

**1. How does tree depth affect overfitting in Decision Trees?**
As `max_depth` increases, the tree keeps splitting on smaller and smaller subsets of training data,
eventually memorizing noise. This is visible in the baseline (unconstrained) tree above, which reaches
near-perfect training accuracy but a noticeably lower test accuracy — the classic train/test gap
signalling overfitting. Restricting depth (as chosen by cross-validation) trades a little training
accuracy for better generalization.

**2. Which hyperparameter had the greatest impact on performance?**
For the Decision Tree, `max_depth` had the largest effect — very shallow trees underfit, very deep or
unbounded trees overfit, and cross-validation accuracy peaks at a moderate depth. For the Random
Forest, `n_estimators` and `max_depth` jointly mattered most: increasing the number of trees stabilizes
performance (variance reduction from averaging), while `max_features` had a comparatively smaller
effect once enough trees were averaged.

**3. How does Random Forest improve generalization?**
Random Forest trains many decision trees on bootstrapped samples (bagging) and considers a random
subset of features at each split. Averaging (majority voting for classification) across many
decorrelated trees cancels out the individual trees' variance, so the ensemble's predictions are more
stable and generalize better than any single deep tree.

**4. Did ensemble learning always improve performance? Why or why not?**
In this experiment, the tuned Random Forest matched or slightly exceeded the tuned Decision Tree's
test accuracy and showed a smaller train/test gap and a higher ROC-AUC. Ensembles usually help most
when the base learner has high variance (deep, unconstrained trees); if the Decision Tree is already
well-regularized via cross-validated hyperparameters, the improvement margin can be small. Random
Forest is not guaranteed to always win — with very small/simple datasets or when the base tree is
already close to optimal, ensembling gives diminishing returns while adding computational and
interpretability cost.


## Conclusion

Decision Tree and Random Forest classifiers were implemented on the Wisconsin Diagnostic Breast
Cancer dataset and evaluated using 5-fold cross-validation. Hyperparameters (`criterion`, `max_depth`,
`min_samples_split`, `min_samples_leaf` for the Decision Tree; `n_estimators`, `max_depth`,
`max_features`, `bootstrap` for the Random Forest) were selected based on average cross-validation
accuracy, ensuring robust generalization rather than overfitting to a single train/test split. The
results confirm that Random Forest reduces variance and improves stability compared to a single
Decision Tree, evidenced by a smaller train/test accuracy gap, more consistent fold-wise accuracy, and
a higher (or comparable) ROC-AUC.
